In [58]:
%pip install xlrd

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 96 kB 9.6 MB/s  eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
from models.models import BertweetClassifier
from preprocessing_steps.data_cleanup import *
from sklearn.model_selection import train_test_split
import pytorch_lightning as pl
import torch
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

class BertweetClassifier(pl.LightningModule):
    def __init__(
        self,
        model_name: str = "vinai/bertweet-base",
        num_labels: int = 3,
        learning_rate: float = 5e-5,
        warmup_ratio: float = 0.10,
        freeze_encoder: bool = True,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.warmup_ratio = warmup_ratio

        # 1) encoder
        self.encoder = SentenceTransformer(model_name)

        # 2) simple linear head
        hidden = self.encoder.get_sentence_embedding_dimension()
        self.classifier = nn.Linear(hidden, num_labels)

        # 3) (optionally) freeze encoder
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

        self.loss_fn = nn.CrossEntropyLoss()

    # Forward expects ONLY the list[str] texts
    def forward(self, texts: list[str]):
        embeds = self.encoder.encode(texts, convert_to_tensor=True, device=self.device)
        return self.classifier(embeds)

    # ------------------------------------------------------------------
    # Shared step to avoid duplication
    def _shared_step(self, batch, stage: str):
        texts, labels = batch                    # <- from our collate_fn
        logits = self(texts)
        loss   = self.loss_fn(logits, labels.to(self.device))

        preds  = logits.argmax(1)
        acc    = (preds == labels.to(self.device)).float().mean()

        # nice logging
        self.log(f"{stage}_loss", loss,  prog_bar=True, on_step=(stage=="train"), on_epoch=True)
        self.log(f"{stage}_acc",  acc,   prog_bar=True, on_epoch=True,   on_step=False)
        # print(f"{stage}_loss", loss,  prog_bar=True, on_step=(stage=="train"), on_epoch=True)
        # print(f"{stage}_acc",  acc,   prog_bar=True, on_epoch=True,   on_step=False)
        return loss


    # Lightning API -----------------------------------------------------
    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        # Re-use the same bookkeeping you used for val:
        self._shared_step(batch, stage="test")
        
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        # `batch` is just a list[str] from your predict_dataloader
        texts = batch
        logits = self(texts)
        probs  = torch.softmax(logits, dim=-1)
        return probs  

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        total_steps  = self.trainer.estimated_stepping_batches
        warmup_steps = int(self.hparams.warmup_ratio * total_steps)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )
        return {
            "optimizer":  optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",      # step every batch
                "frequency": 1,
                "name": "linear-warmup",
            },
        }
    
def get_dataloader_labeled(df=None, text_column='clean_tweet', target=None, shuffle=True, batch_size=64, drop_last=True):
  data = list(zip(df[text_column].tolist(), df[target].tolist()))
  dataloader = DataLoader(df, shuffle=shuffle, batch_size=batch_size,
                          collate_fn=lambda x:x, drop_last=drop_last)
  return dataloader


class TweetsDataModule(pl.LightningDataModule):
    def __init__(self, data: pd.DataFrame, batch_size: int = None, target_col: str = 'AR'):
        super().__init__()
        self.data = data.copy()
        self.batch_size = batch_size
        self.target_col = target_col

        self.label2id: dict[str, int] = {}
        self.id2label: list[str]      = []
    
    def setup(self, stage: str=None):
        # Assign train/val datasets for use in dataloaders
        self.data = preprocess_text(self.data, text_col="text")
        # self.data = self.data[['clean_text', self.target_col]]

        uniques          = sorted(self.data[self.target_col].unique())
        self.label2id    = {lbl: i for i, lbl in enumerate(uniques)}
        self.id2label    = uniques                            # same order
        self.data["y"]   = self.data[self.target_col].map(self.label2id)    

        train, test = train_test_split(self.data, train_size=0.8, random_state=2025, shuffle=True)
        self.train, self.val = train_test_split(train, train_size=0.8, random_state=2025, shuffle=True)
        self.test = test


    def train_dataloader(self):
        #  NO super().__init__() here
        data = list(zip(self.train["clean_text"], self.train["y"]))
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=lambda batch: ( [t for t, _ in batch],
                                        torch.tensor([l for _, l in batch]) ),
            drop_last=True,
        )

    def val_dataloader(self):
        data = list(zip(self.val["clean_text"], self.val["y"]))
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=lambda batch: ( [t for t, _ in batch],
                                        torch.tensor([l for _, l in batch]) ),
        )

    def test_dataloader(self):
        data = list(zip(self.test["clean_text"], self.test["y"]))
        return DataLoader(
            data,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=lambda batch: ( [t for t, _ in batch],
                                        torch.tensor([l for _, l in batch]) ),
        )
    
    def predict_dataloader(self):
        """
        Returns a DataLoader for inference.
        - If you called `setup("predict")`, it will use `self.test`.
        - If you created `self.predict` manually (e.g., a new DataFrame without labels),
          that takes precedence.
        Collate fn yields a *list[str]* so the model’s forward just receives raw texts.
        """
        dataset = getattr(self, "predict", None)
        if dataset is None:          # fall back to the held-out test set
            dataset = self.test

        texts = list(dataset["clean_text"])

        return DataLoader(
            texts,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=lambda batch: batch  # returns List[str] per batch
        )


tweets_labeled = pd.read_csv('data/training_data.csv')
tweets_labeled['AR'].replace({4:2}, inplace=True)
dataModule = TweetsDataModule(tweets_labeled, 32)
dataModule.setup()
model_name = "digio/Twitter4SSE"
model_name2 = "peulsilva/sentence-transformer-trained-tweet"
model = BertweetClassifier(
    model_name=model_name2,
    learning_rate=1e-4,
    num_labels=len(dataModule.id2label)           # keep model & data in sync
)
model.id2label = dataModule.id2label          # list   -> e.g. ["neg","neu","pos"]
model.label2id = dataModule.label2id
trainer = pl.Trainer(max_epochs=1)
trainer.fit(model=model, datamodule=dataModule)


/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_21080/2023251847.py:176: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  tweets_labeled['AR'].replace({4:2}, inplace=True)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Loading `train_dataloader` to estimate number of stepping batches.
/Users/alexa

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 74/74 [01:07<00:00,  1.10it/s, v_num=45, train_loss_step=1.040, val_loss=1.070, val_acc=0.388, train_loss_epoch=1.130, train_acc=0.328]

`Trainer.fit` stopped: `max_epochs=1` reached.


Epoch 0: 100%|██████████| 74/74 [01:08<00:00,  1.08it/s, v_num=45, train_loss_step=1.040, val_loss=1.070, val_acc=0.388, train_loss_epoch=1.130, train_acc=0.328]


In [12]:
# 3epoch 5e-5lr: [{'test_loss': 0.43339282274246216, 'test_acc': 0.35087719559669495}]
# 5epoch 5e-5lr: [{'test_loss': 0.4054107964038849, 'test_acc': 0.36572200059890747}]
# 5epoch 1e-4lr: [{'test_loss': 0.4054107964038849, 'test_acc': 0.36572200059890747}]

test_metrics = trainer.test(model=model, datamodule=dataModule)      # <- returns a list of dicts
print(test_metrics)

/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 24/24 [00:15<00:00,  1.55it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.43589743971824646    │
│         test_loss         │     1.073567271232605     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.073567271232605, 'test_acc': 0.43589743971824646}]


In [57]:
!pip install matplotlib

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [9]:
from sklearn.metrics import confusion_matrix,  classification_report
# import matplotlib.pyplot as plt
import torch
import numpy as np

# ------------------------------------------------------------
# 1. Run inference -> stacked probability tensor
# ------------------------------------------------------------
# preds = trainer.predict(model=model, datamodule=dataModule)

flat  = [tb for dev in preds for tb in (dev if isinstance(dev, list) else [dev])]
probs = torch.cat(flat).cpu().numpy()            # shape [N_samples, num_labels]

# ------------------------------------------------------------
# 2. Turn probs into predicted class IDs & labels
# ------------------------------------------------------------
pred_idx    = probs.argmax(axis=1)               # int IDs 0 … C-1
pred_labels = [dataModule.id2label[i] for i in pred_idx]

# ------------------------------------------------------------
# 3. Assemble a tidy DataFrame
# ------------------------------------------------------------
df_pred = dataModule.test.reset_index(drop=True).copy()
df_pred[[f"p_{lbl}" for lbl in dataModule.id2label]] = probs
df_pred["pred_idx"] = pred_idx
df_pred["pred"]     = pred_labels     # <- string label—for reports, plots, etc.

# ------------------------------------------------------------
# 4. Confusion matrix (true vs. predicted IDs)
# ------------------------------------------------------------
cm = confusion_matrix(
    df_pred["y"],            # true integer IDs stored earlier
    df_pred["pred_idx"],
    labels=np.arange(len(dataModule.id2label))
)

print(classification_report(df_pred["y"],df_pred["pred_idx"]))


/Users/alexandermichaeltjhin/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 24/24 [00:23<00:00,  1.03it/s]
              precision    recall  f1-score   support

           0       0.41      0.55      0.47       258
           1       0.20      0.07      0.10       162
           2       0.50      0.53      0.51       321

    accuracy                           0.44       741
   macro avg       0.37      0.38      0.36       741
weighted avg       0.40      0.44      0.41       741



In [11]:
df_pred.head()

,Unnamed: 0,tweet_id,text,author_id,tw_date,year,AR,MB,source,split,clean_text,y,p_1,p_2,p_3,pred_idx,pred
0,1,7.300000e+17,Met w/folks from General Aviation Mfg Assoc. a...,1061029050,2016-05-11 00:00:00,2016,2,2,randomized_2016,train,Met with/folks from General Aviation Mfg Assoc...,1,0.498821,0.190423,0.310755,0,1
1,1470,1.070000e+18,"No longer just Cohen versus Trump. AMI, the pa...",278124059.0,2018-12-12 00:00:00,2018,1,1,randomized_2018,train,"No longer just Cohen versus Trump. AMI, the pa...",0,0.366830,0.195357,0.437813,2,3
2,906,9.098001e+17,We won't kick 800k DREAMers out of the only ho...,Catherine Cortez Masto,2017-09-18 15:23:49+00:00,2017,2,2,selected_2017,NaN,We won't kick 800k DREAMers out of the only ho...,1,0.363065,0.300001,0.336934,0,1
3,791,1.370000e+18,Don't let Republicans of 👉Congress👈 tell you t...,328679423.0,2021-03-22 00:00:00,2021,2,2,randomized_2021,train,Don't let Republicans of Congress tell you the...,1,0.299461,0.317223,0.383316,2,3
4,107,1.080000e+18,RT @13JaclynLee: The Ashanti Alert is official...,Mark Warner,2019-01-02 14:31:28+00:00,2019,2,2,selected_2019,NaN,retweet 13JaclynLee: The Ashanti Alert is offi...,1,0.355932,0.344531,0.299537,0,1


In [ ]:
def get_labels(df, col_name):
    conditions = [
        df[col_name] == 0,
        df[col_name] == 1,
        df[col_name] == 2
    ]
    choices = ['Problem', 'Solution', 'Other']
    return np.select(conditions, choices, default=np.NAN)

# Data process

In [104]:
import os
path = 'data/RandomSampledTweets/'

dfs = []
for file in os.listdir(path):
    print(file)
    source = "randomized_" + file[-13:-9]
    if 'Done' in file:
        full_path = os.path.join(path,file)
        df = pd.read_csv(full_path)
        df = df[['tweet_id', 'text', 'author_id', 'tw_date', 'year', 'AR', 'MB']]
        df['tw_date'] = pd.to_datetime(df['tw_date'])
        df['author_id'] = df['author_id'].astype(str)
        df['source'] = source
        # df['tweet_id'] = df['tweet_id'].astype(str)
        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data.head()

random_tweets_2016_Done.csv
random_tweets_2021_Done.csv
random_tweets_2014_Done.csv
random_tweets_2015_Done.csv
random_tweets_2018_Done.csv
random_tweets_2013_Done.csv
random_tweets_2012_Done.csv


,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,7.300000e+17,Congress has responsibility to protect familie...,1061029050,2016-05-11,2016,1,1,randomized_2016
1,7.300000e+17,Met w/folks from General Aviation Mfg Assoc. a...,1061029050,2016-05-11,2016,2,2,randomized_2016
2,7.080000e+17,Talked about the benefits of bicycling for ND ...,1061029050,2016-03-09,2016,3,3,randomized_2016
3,7.820000e+17,At the Dick Morris Memorial Tailgate for the U...,10615232,2016-10-01,2016,3,3,randomized_2016
4,8.040000e+17,SOON: I will speak on the #SenateFloor on the ...,1071402577,2016-11-30,2016,3,3,randomized_2016


In [123]:
data['split'] = 'train'  # Default all to train

# Group by year and sample 50 for the test set
test_indices = (
    data.groupby('year', group_keys=False)
      .apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
      .index
)
data.loc[test_indices, 'split'] = 'test'

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_20388/3707914531.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data.groupby('year', group_keys=False)


In [129]:
data_train = data[data['split']=='train']

In [127]:
test_data = data[data['split']=='test']
test_data.drop('split',axis=1, inplace=True)
test_data.head()

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_20388/3092706159.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data.drop('split',axis=1, inplace=True)


,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,7.300000e+17,Congress has responsibility to protect familie...,1061029050,2016-05-11,2016,1,1,randomized_2016
5,8.070000e+17,"Last night, my CHIP IN for Vets bill was calle...",1071402577,2016-12-08,2016,2,2,randomized_2016
9,8.100000e+17,Great food/convo about north Omaha biz &amp; g...,1071402577,2016-12-16,2016,1,3,randomized_2016
15,7.550000e+17,Talking about strategies to keep #AsianCarp ou...,1074518754,2016-07-19,2016,1,1,randomized_2016
22,7.970000e+17,"Today, I met with local veterans &amp; service...",1099199839,2016-11-10,2016,3,3,randomized_2016


In [134]:
test_data.year.value_counts()

year
2016    50
2021    50
2014    50
2015    50
2018    50
2013    50
2012    50
Name: count, dtype: int64

In [128]:
test_data.to_csv("data/test_data.csv")

In [108]:
str(data['tweet_id'].tolist()[0])

'7.3e+17'

In [120]:
import numpy as np
df_path = './data/MichelleCoding1500 corrected march_13_2025.xlsx'
df_1500 = pd.read_excel(df_path)
df_1500['date'] = pd.to_datetime(df_1500['date'])
df_1500 = df_1500[['id', 'text', 'author_id', 'date', 'year', 'AR', 'MB']]
df_1500.rename(columns={'id':'tweet_id', 'date':'tw_date'}, inplace=True)
df_1500['source'] = np.where(df_1500['year'] == 2020, 'selected_2020', 'selected_2017')
df_1500.head()

,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,908481742163517056,Please join in building support and momentum f...,Lindsey Graham,2017-09-15 00:05:00+00:00,2017,2,2,selected_2017
1,908482907320339968,.@FLOIR_comm has updated its ‚ÄúHurricane Seas...,Rick Scott,2017-09-15 00:09:38+00:00,2017,2,2,selected_2017
2,908483151206509952,RT @HealthyFla: #FLHealth urges caution with s...,Rick Scott,2017-09-15 00:10:36+00:00,2017,1,1,selected_2017
3,908483250032692992,Kim Jong-un's missile launches show how much h...,James Lankford,2017-09-15 00:11:00+00:00,2017,1,1,selected_2017
4,908483923109448960,@realDonaldTrump's refusal to hold white supre...,Bob Casey,2017-09-15 00:13:40+00:00,2017,1,1,selected_2017


In [139]:
df_1500['split'] = 'train'  # Default all to train

# Group by year and sample 50 for the test set
test_indices = (
    df_1500.groupby('year', group_keys=False)
      .apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
      .index
)
df_1500.loc[test_indices, 'split'] = 'test'
df_1500_train = df_1500[df_1500['split']=='train'].drop('split', axis=1)
df_1500_test = df_1500[df_1500['split']=='test'].drop('split', axis=1)

/var/folders/_5/c3vpzww13z34kn9h6pvd032h0000gn/T/ipykernel_20388/1607151548.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_1500.groupby('year', group_keys=False)


In [109]:
df_2019 = pd.read_excel('data/2019_updated.xls')
df_2019['id'] = df_2019['id']
df_2019['created_at'] = pd.to_datetime(df_2019['created_at'])
df_2019['year'] = df_2019['created_at'].dt.year
df_2019 = df_2019[['id', 'text', 'author_id', 'created_at', 'year', 'AR', 'MB']]
df_2019.rename(columns={'id':'tweet_id', 'created_at':'tw_date'}, inplace=True)
df_2019['source'] = 'selected_2019'
df_2019.head()

,tweet_id,text,author_id,tw_date,year,AR,MB,source
0,1.080000e+18,Wishing everyone a safe #NewYearsEve and a hap...,Bob Menendez,2019-01-01 00:00:21+00:00,2019,3,3,selected_2019
1,1.080000e+18,Just a few. I am sure others have their nomine...,John Cornyn,2019-01-01 00:00:27+00:00,2019,3,3,selected_2019
2,1.080000e+18,IÇƒÙm proud of what weÇƒÙve accomplished for N...,Catherine Cortez Masto,2019-01-01 00:14:50+00:00,2019,2,2,selected_2019
3,1.080000e+18,"As your next senior senator, I enter the 116th...",Catherine Cortez Masto,2019-01-01 00:15:15+00:00,2019,2,1,selected_2019
4,1.080000e+18,"Out of a tragedy, some great news: with the Pr...",Mark Warner,2019-01-01 00:59:26+00:00,2019,2,2,selected_2019


In [143]:
df_2019['split'] = 'train'  # Default all to train

# Group by year and sample 50 for the test set
test_indices = (
    df_2019.apply(lambda x: x.sample(n=min(50, len(x)), random_state=42))
      .index
)
df_2019.loc[test_indices, 'split'] = 'test'
df_2019_train = df_2019[df_2019['split']=='train'].drop('split', axis=1)
df_2019_test = df_2019[df_2019['split']=='test'].drop('split', axis=1)

In [145]:
all_test=pd.concat([test_data, df_2019_test, df_1500_test])

In [148]:
all_test.year.value_counts().sort_index()
all_test.to_csv('data/test_data.csv')

In [149]:
df_all_labeled = pd.concat([data_train, df_2019_train, df_1500_train])

In [150]:
df_all_labeled.source.value_counts().sort_index()

source
randomized_2012    121
randomized_2013    127
randomized_2014    224
randomized_2015    206
randomized_2016    316
randomized_2018    418
randomized_2021    408
selected_2017      951
selected_2019      484
selected_2020      449
Name: count, dtype: int64

In [151]:
df_all_labeled.year.value_counts().sort_index()

year
2012    121
2013    127
2014    224
2015    206
2016    316
2017    951
2018    418
2019    484
2020    449
2021    408
Name: count, dtype: int64

In [152]:
df_all_labeled.to_csv('data/training_data.csv')

In [111]:
data.shape

(2170, 8)

In [113]:
df_1500.year.value_counts().sort_index()

year
2017    1001
2020     499
Name: count, dtype: int64